In [ ]:
# get_symbols.py
# 抓 S&P500 並維護 sp500_symbols table（簡單版，直接使用網站回傳的 symbol）
import pandas as pd
import datetime
from typing import List
from sqlalchemy import select, update, Table, Column, MetaData, Boolean, String, DateTime
from database import get_engine

def get_sp500_symbols_from_web() -> List[str]:
    """從 stockanalysis.com 抓 S&P500 list，直接回傳 Symbol 欄位內容"""
    url = "https://stockanalysis.com/list/sp-500-stocks/"
    tables = pd.read_html(url)
    df_sp500 = tables[0]
    return df_sp500['Symbol'].tolist()

def get_symbols_table(engine):
    """建立 sp500_symbols table（若不存在）"""
    metadata = MetaData()
    table = Table(
        "sp500_symbols", metadata,
        Column("symbol", String, primary_key=True),
        Column("status", Boolean, nullable=False),
        Column("date_added", DateTime),
        Column("date_removed", DateTime),
    )
    metadata.create_all(engine)
    return table

def update_symbols():
    """
    同步 S&P500（每日跑一次），**直接使用網站回傳的 symbol**（假設與 API 一致）
    流程：
      1) 抓 web raw symbols
      2) 與 DB 做差異（新增 / 恢復 / 標記移除）
    """
    engine = get_engine()
    table = get_symbols_table(engine)
    raw_symbols = get_sp500_symbols_from_web()
    today = datetime.date.today()

    # 直接把網站回傳的 symbol 當作 canonical list
    canonical_set = set(raw_symbols)

    with engine.begin() as conn:
        # 讀現有 DB symbols（只取 symbol 欄）
        rows = conn.execute(select(table.c.symbol)).fetchall()
        existing_symbols = {r[0] for r in rows}

        # 新增或恢復：canonical_set 裡有但 DB 冇
        for sym in canonical_set:
            if sym not in existing_symbols:
                conn.execute(table.insert().values(symbol=sym, status=True, date_added=today))
            else:
                # 若已存在但 status=False，恢復為 True
                row = conn.execute(select(table).where(table.c.symbol == sym)).fetchone()
                if row and not row['status']:
                    conn.execute(update(table).where(table.c.symbol == sym).values(status=True, date_added=today, date_removed=None))

        # 標記移除：DB 裡有但唔喺 canonical_set（代表已被移除）
        for sym in list(existing_symbols):
            if sym not in canonical_set:
                row = conn.execute(select(table).where(table.c.symbol == sym)).fetchone()
                if row and row['status']:
                    conn.execute(update(table).where(table.c.symbol == sym).values(status=False, date_removed=today))

    print("update_symbols done. total symbols:", len(canonical_set))

def get_active_symbols() -> List[str]:
    """回傳目前 status=True 的 symbol list"""
    engine = get_engine()
    metadata = MetaData()
    table = Table("sp500_symbols", metadata, autoload_with=engine)
    with engine.connect() as conn:
        rows = conn.execute(select(table).where(table.c.status == True)).fetchall()
    return [r['symbol'] for r in rows]

if __name__ == "__main__":
    update_symbols()
